# 1장 3강: 선형 변환과 신경망 레이어 — 실습문제

## 실습 목표

- 같은 위치끼리 곱해 더하여 선형 변환을 계산합니다.
- 편향을 더하는 아핀 변환과 선형 변환의 차이를 확인합니다.
- 가중치와 편향을 각각 바꿀 때 점수가 어떻게 달라지는지 확인합니다.
- ReLU의 규칙을 적용하고, 두 레이어의 계산을 순서대로 따라갑니다.

## 실습 환경 / 데이터

- Python, NumPy를 사용합니다. 외부 데이터 파일은 필요하지 않습니다.
- 제공된 「1장 3강: 선형 변환과 신경망 레이어」의 예제와 연습문제에 있는 숫자·방법을 사용합니다.
- 새로운 공식을 유도하거나 모델을 학습시키는 과제는 없습니다.

## 실습 구성

| 구분 | 내용 | 교안 근거 |
|---|---|---|
| 필수 1 / 문제 1-1 | 선형·아핀 변환과 0벡터 | 1절의 x=[3,2] 예제 |
| 필수 1 / 문제 1-2 | 가중치와 편향을 따로 변경 | 2절·연습문제 2 |
| 필수 2 / 문제 2-1 | ReLU와 한 레이어 계산 | 3절의 고객 3명 예제 |
| 필수 2 / 문제 2-2 | 은닉층에서 출력층으로 전달 | 3절의 전체 순전파 예제 |
| 과제 / 문제 3-1 | 선형·아핀 변환 반복 | 교안 연습문제 1 |
| 과제 / 문제 3-2 | 두 레이어 순전파 반복 | 교안 연습문제 3 |

필수 실습 4문제와 기초 과제 2문제입니다. 과제도 필수 실습에서 사용한 계산 순서를 그대로 적용합니다.

## 진행 안내

1. 위에서 아래로 읽고, 먼저 NumPy를 불러오는 셀을 실행합니다.
2. 각 문제의 숫자와 작성 도움말을 보고 코드를 한 줄씩 작성합니다.
3. 출력값을 확인한 뒤 답변 작성란에 계산 과정과 답을 적습니다.
4. 수식은 먼저 말로 읽고, 곱셈 → 덧셈 → 편향 → ReLU 순서를 확인합니다. ReLU는 적용하라고 한 곳에서만 사용합니다.
5. 변수는 각 문제에서 다시 만듭니다. 앞 문제의 입력값이나 가중치를 그대로 쓰지 않도록 주의하세요.

## 처음 보는 용어와 기호

| 용어·기호 | 쉬운 뜻 |
|---|---|
| 입력 / 출력 | 계산에 넣는 숫자 / 계산해서 얻는 숫자 |
| 벡터 / 행렬 | 숫자를 한 줄로 나열한 것 / 숫자를 행과 열로 놓은 것 |
| 행 / 열 | 가로로 읽는 줄 / 세로로 읽는 줄 |
| 특성 | 고객을 나타내는 한 가지 정보. 예: 방문 횟수 |
| x / X | 한 고객의 입력 / 여러 고객의 입력을 모은 배열 |
| w / W | 입력값에 곱할 가중치. W의 각 열은 출력 하나에 사용할 가중치 |
| b / bias | 곱하고 더한 결과에 마지막으로 더할 편향 |
| 파라미터 | 이 강의에서는 가중치와 편향을 가리키는 말 |
| 뉴런 / 레이어 | 입력으로 출력 하나를 만드는 계산 단위 / 뉴런들을 모은 계산 단계 |
| 은닉층 / 출력층 | 입력과 최종 결과 사이의 계산 단계 / 최종 결과를 만드는 단계 |
| Z, Z1 | 가중치를 곱해 더하고 편향까지 더한 값 |
| A, H | ReLU를 적용한 값. 고객 이름 A와 코드 변수 A는 문맥으로 구분 |
| Y | 최종 출력값을 모은 배열 |
| Shape | 배열의 모양. (3,2)는 3행 2열, (2,)는 숫자 2개인 1차원 배열 |

`xW`는 수식에서 x와 W의 행렬곱을 뜻하며, 코드에서는 `x @ W`로 씁니다. `xW+b`는 그 결과에 편향 b를 더하라는 뜻입니다. `T(x)`는 ‘입력 x에 T라는 계산을 적용한 결과’로 읽습니다.

## 실습 준비

아래 셀을 먼저 실행하세요. NumPy는 배열을 계산하는 도구이며, `np`는 코드에서 NumPy를 짧게 부르는 이름입니다.

In [1]:
import numpy as np

---

## 필수 1. 선형·아핀 변환과 파라미터 이해하기

### 문제 1-1. 선형 변환에 편향을 더해 보기

#### 문제 설명

교안 1절의 예제입니다. 선형 변환은 입력에 가중치 행렬을 곱하는 계산이며, 아핀 변환은 그 결과에 편향을 더하는 계산입니다.

```python
x = np.array([3, 2])
W = np.array([[2, 0], [0, 3]])
bias = np.array([1, -2])
zero = np.array([0, 0])
```

- 선형 변환: `x @ W`
- 아핀 변환: `x @ W + bias`

W의 첫 번째 열은 [2,0], 두 번째 열은 [0,3]입니다. 0벡터는 모든 값이 0인 벡터를 말합니다.

#### 요구사항

1. x, W, bias, zero를 위와 같이 만드세요.
2. `linear_output = x @ W`, `affine_output = x @ W + bias`를 계산하여 출력하세요.
3. `zero @ W`와 `zero @ W + bias`를 출력하세요.
4. 답변 작성란에 두 출력값의 곱셈·덧셈 과정을 적으세요.

#### 작성 도움말

첫 출력은 x의 [3,2]와 W의 첫 열 [2,0]을 같은 순서로 짝지어 `3×2 + 2×0`으로 계산합니다. 둘째 출력도 둘째 열을 사용해 같은 방법으로 계산하세요. 편향은 첫 출력에 1, 둘째 출력에 −2를 각각 더합니다.

#### 해석 질문

**Q1.** 선형 변환의 첫 번째·두 번째 출력은 각각 어떻게 계산되나요?

**Q2.** 편향을 더한 아핀 변환 결과는 무엇인가요?

**Q3.** 0벡터 입력에서 두 변환의 결과는 각각 무엇인가요?

**Q4.** 편향이 [0,0]이라면 두 변환의 결과가 같아지나요?

#### 제출 결과

- 두 변환의 출력과 0벡터의 출력
- 계산 과정과 Q1~Q4 답변

In [ ]:
# 문제 1-1. 선형 변환에 편향을 더해 보기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.



#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 선형 변환의 첫 번째·두 번째 출력은 각각 어떻게 계산되나요?**

- 답변: 첫 출력은 W의 첫 열 [2,0]을 사용해 3*2+2*0=6+0=6
- 둘째 출력은 둘쨰 열 [0,3]을 사용해 3*0+2*3=0+6=6
- 두 결과를 순서대로 모으면 [6,6]이다 
- 입력의 첫 값과 열의 첫 값, 입력의 둘째 값과 열의 둘째 값을 짝지어 곱한다


**Q2. 편향을 더한 아핀 변환 결과는 무엇인가요?**

- 답변: 선형 결과 [6,6]에 편향 [1,-2]를 같은 위치끼리 더한다
- 첫 값은 6+1=7 둘째 값은 6+(-2)=4이므로 아핀 결과는 [7,4]이다

**Q3. 0벡터 입력에서 두 변환의 결과는 각각 무엇인가요?**

- 답변: 선형 결과는 [0,0]이다
- 첫 값은 0*2+0*0=0 둘째 값은 0*0+0*3=0 이기 떄문이다
- 아핀 결과는 그 값에 편향을 더한 [0+1,0-2]=[1,-2]이다
- 입력이 모두 0이면 가중합은 0이지만 평향은 남는다

**Q4. 편향이 [0,0]이라면 두 변환의 결과가 같아지나요?**

- 답변: 같아진다
- 선형ㅇ 결과 [6,6]에 [0,0]을 더해도 [6,6]이기 때문이다
- 이번 두 식에서 다른 부분은 편향 덧셈뿐이므로 편향이 모두 0이면 계산결과도 같다

### 문제 1-2. 가중치와 편향을 하나씩 바꾸기

#### 문제 설명

교안 연습문제 2의 데이터입니다. 각 고객의 첫 입력은 방문 횟수, 둘째 입력은 장바구니 추가 횟수입니다.

```python
X = np.array([[4, 2], [1, 5]])
w = np.array([2, -1])
b = 3
```

X의 첫 행 [4,2]는 고객 A, 둘째 행 [1,5]는 고객 B입니다. 고객별 점수는 ‘첫 입력×첫 가중치 + 둘째 입력×둘째 가중치 + 편향’으로 계산합니다.

| 비교 | 가중치 | 편향 |
|---|---|---|
| 기본 | [2,−1] | 3 |
| 가중치만 변경 | [2,−2] | 3 |
| 편향만 변경 | [2,−1] | 5 |

두 변경은 각각 기본 설정에서 출발합니다.

#### 요구사항

1. 위 배열을 만들고 `base_scores = X @ w + b`를 계산하여 출력하세요.
2. `w_new = np.array([2, -2])`로 만들고 `weight_scores = X @ w_new + b`를 계산하세요. 점수와 `weight_scores - base_scores`를 출력하세요.
3. 기본 w를 사용하여 `b_new = 5`, `bias_scores = X @ w + b_new`를 계산하세요. 점수와 `bias_scores - base_scores`를 출력하세요.
4. `zero = np.array([0, 0])`를 만들고 기본 설정인 `zero @ w + b`를 출력하세요.

#### 작성 도움말

변화량은 ‘변경 후 점수 − 기본 점수’입니다. 변화량이 −2라면 점수가 2만큼 줄었다는 뜻입니다. 편향만 바꿀 때는 w_new가 아닌 기본 w를 사용하세요. 한 번에 한 조건만 바꿉니다.

#### 해석 질문

**Q1.** 기본 설정에서 고객 A와 B의 점수는 각각 얼마인가요?

**Q2.** 가중치만 바꾸면 각 점수와 변화량은 얼마인가요?

**Q3.** 편향만 바꾸면 각 점수와 변화량은 얼마인가요?

**Q4.** 기본 설정에 0벡터를 넣으면 점수가 얼마인가요? 왜 그 값이 남나요?

#### 제출 결과

- 기본 점수, 두 변경의 점수와 변화량, 0벡터의 점수
- 계산 과정과 Q1~Q4 답변

In [8]:
# 문제 1-2. 가중치와 편향을 하나씩 바꾸기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.

X = np.array([
    [4,2], 
    [1,5]
    ])

w = np.array([ 2, -1])
b = 3
base_scores = X @ w + b
print("base_scores:", base_scores)

w_new = np.array([ 2, -2])
weight_scores = X @ w_new + b
print("weight_scores:", weight_scores)
print("weight_scores - base_scores:", weight_scores - base_scores)

b_new = 5
bias_scores = X @ w + b_new
print("bias_scores:", bias_scores)
print("bias_scores - base_scores:", bias_scores - base_scores)

zero = np.array([0, 0])
print("기본 설정의 0벡터 출력: ", zero @ w + b)

base_scores: [9 0]
weight_scores: [ 7 -5]
weight_scores - base_scores: [-2 -5]
bias_scores: [11  2]
bias_scores - base_scores: [2 2]
기본 설정의 0벡터 출력:  3


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 기본 설정에서 고객 A와 B의 점수는 각각 얼마인가요?**

- 답변: A는 4*2+2*(-1)+3=8-2+3=9 이다 B는 1*2+5*(-1)+3=0이다
- 따라서 기본 점수 배열은 [9,0]이다
- 곱셈을 먼저 하고 그 결과들과 편향을 더한다

**Q2. 가중치만 바꾸면 각 점수와 변화량은 얼마인가요?**

- 답변: A는 4*2+2*(-2)+3=8-4+3=7 이다 변화량은 7-9=-2
- B는 1*2+5*(-2)+3=-5 이다 변화량은 -5-0 = -5이다
- 점수는 [7,-5], 변화량은 [-2,-5]이다
- 둘째 입력이 각각 2와 5이므로 같은 가중치 변화도 두 고객에게 다르게 반영된다

**Q3. 편향만 바꾸면 각 점수와 변화량은 얼마인가요?**

- 답변: 가중치는 기본 [2,-1]을 사용한다 A는 8-2+5=11 B는 2-5+5=2 이다
- 변화량은 A가 11-9=2, Brk 2-0=2이므로 [2,2]이다
- 편향이 3에서 5로 2 커졌고 가중합은 그대로라서 두 점수에 똑같이 2가 더해진다

**Q4. 기본 설정에 0벡터를 넣으면 점수가 얼마인가요? 왜 그 값이 남나요?**

- 답변: 점수는 3이다 0 * 2 + 0 * (-1) + 3 = 0 + 0 + 3= 3으로 계산한다
- 두 입력이 모두 0이라 가중치를 곱한 값이 0이 되고 마지막에 더하는 평향 3만 남기 때문이다

---

## 필수 2. ReLU와 신경망 순전파

### 문제 2-1. ReLU와 한 레이어를 순서대로 계산하기

#### 문제 설명

교안 3절의 예제입니다. 먼저 ReLU의 규칙을 연습한 뒤 레이어 하나를 계산합니다.

**ReLU(렐루)의 규칙:** 양수는 그대로 두고, 음수는 0으로 바꿉니다. 0은 0으로 둡니다.

```python
def relu(z):
    return np.maximum(0, z)

test_z = np.array([-3, -1, 0, 2, 5])
X = np.array([[3, 2], [1, 4], [0, 2]])
W = np.array([[1, -2], [2, 1]])
b = np.array([-4, 1])
```

X의 세 행은 고객 A, B, C입니다. W의 두 열은 뉴런 두 개의 가중치입니다. 첫 뉴런의 가중치는 [1,2], 둘째 뉴런의 가중치는 [−2,1]입니다. 편향은 각각 −4와 1입니다.

이번 계산은 **가중치 적용 → 편향 덧셈 → ReLU**의 세 단계입니다.

#### 요구사항

1. 위 코드를 입력하고 `relu(test_z)`를 출력하세요.
2. `weighted = X @ W`를 계산하여 출력하세요.
3. `Z = weighted + b`를 계산하여 출력하세요.
4. `A = relu(Z)`를 계산하여 출력하세요.
5. `Z.shape`와 `A.shape`를 출력하고, 고객 A의 두 뉴런 계산을 손으로 확인하세요.

#### 작성 도움말

`def relu(z):`와 그 아래 `return` 줄은 위 예제를 그대로 사용하세요. return 앞의 들여쓰기도 유지하세요. 고객 A의 첫 뉴런은 `3×1 + 2×2`를 구한 뒤 −4를 더합니다. 마지막에 그 값과 0 중 큰 값을 고릅니다.

#### 해석 질문

**Q1.** ReLU는 [−3,−1,0,2,5]를 어떻게 바꾸나요?

**Q2.** 고객 A의 가중합, 편향을 더한 값 Z, ReLU 결과 A는 각각 무엇인가요?

**Q3.** 고객 B와 C의 Z와 ReLU 결과는 각각 무엇인가요?

**Q4.** Z와 A의 Shape (3,2)는 무엇을 뜻하나요? 음수를 0으로 바꾸면 배열의 크기도 줄어드나요?

#### 제출 결과

- ReLU 연습 결과, weighted·Z·A의 출력, Z·A의 Shape
- 고객 A의 계산 과정과 Q1~Q4 답변

In [9]:
# 문제 2-1. ReLU와 한 레이어를 순서대로 계산하기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.

def relu(z):
    return np.maximum(0, z)

test_z = np.array([-3, -1, 0, 2, 5])
print("테스트 입력", test_z)
print("테스트 ReLU", relu(test_z))

X = np.array([
    [3,2],
    [1,4],
    [0,2]
])

W = np.array([
    [1,-2],
    [2,1]
])

b = np.array([-4,1])
print("X shape:", X.shape)
print("W shape:", W.shape)
print("b shape:", b.shape)

weighted = X @ W
Z = weighted + b
A = relu(Z)
print("편향 전 행력곱:", weighted)
print("편향을 더한:", Z)
print("ReLU 결과:", A)
print("Z shape:", Z.shape)
print("A shape:", A.shape)


테스트 입력 [-3 -1  0  2  5]
테스트 ReLU [0 0 0 2 5]
X shape: (3, 2)
W shape: (2, 2)
b shape: (2,)
편향 전 행력곱: [[ 7 -4]
 [ 9  2]
 [ 4  2]]
편향을 더한: [[ 3 -3]
 [ 5  3]
 [ 0  3]]
ReLU 결과: [[3 0]
 [5 3]
 [0 3]]
Z shape: (3, 2)
A shape: (3, 2)


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. ReLU는 [−3,−1,0,2,5]를 어떻게 바꾸나요?**

- 답변: [0,0,0,2,5]로 바꾼다. -3과 -1은 0보다 작으므로 0, 입력 0은 그대로 0,2와 5는 0보다 크므로 그대로 남는다
- 음수를 양수로 바꾸는 것이 아닌 0으로 바꾸는 것이다

**Q2. 고객 A의 가중합, 편향을 더한 값 Z, ReLU 결과 A는 각각 무엇인가요?**

- 답변: 가중합은 [7,-4]이다 첫 뉴런은 3*1+2*2=7, 둘째 뉴런은 3*(-2)+2*1=-4이다
- 편향을 더하면 Z는 [7-4,-4+1]=[3,-3]이다
- ReLU는 3을 유지하고 -3을 0으로 바꾸므로 A 배열에서 이 고객의 행은 [3,0]이다

**Q3. 고객 B와 C의 Z와 ReLU 결과는 각각 무엇인가요?**

- 답변: B의 첫 값은 1*1+4*2-4=5 둘째 값은 1*(-2)+4*1+1=3 이므로 Z=[5,3]이다
- 둘 다 양수라서 ReLU 후에도 [5,3]이다
- C는 0*1+2*2-4=0, 0*(-2)+2*1+1=3이므로 Z=[0,3]이다 0과 양수는 그대로여서 ReLU 후에도 [0,3]이다

**Q4. Z와 A의 Shape (3,2)는 무엇을 뜻하나요? 음수를 0으로 바꾸면 배열의 크기도 줄어드나요?**

- 답변: 3은 고객 3명을 나타내는 행의 수이고, 2는 고객마다 뉴런 두 개가 만든 출력의 수이다
- Z와 A 모두 Shape가 (3,2)이다
- ReLU는 음수인 자리에 숫자 0을 넣을 뿐 그 자리를 지우지 않으므로 배열의 크기는 줄지 않는다

### 문제 2-2. 은닉층의 결과를 출력층에 전달하기

#### 문제 설명

교안 3절의 전체 순전파 예제입니다. 순전파란 입력부터 출력까지 앞 단계의 결과를 다음 단계로 전달하며 계산하는 과정입니다.

```python
X = np.array([[3, 2], [1, 4], [0, 2]])
W1 = np.array([[1, -2], [2, 1]])
b1 = np.array([-4, 1])
W2 = np.array([[2], [-1]])
b2 = np.array([1])
```

첫 단계인 은닉층은 문제 2-1과 같습니다. 은닉층의 뉴런은 2개이고, 그 뒤에 뉴런 1개인 출력층을 붙입니다. 아래 세 줄을 위에서부터 계산하세요.

```python
Z1 = X @ W1 + b1
H = relu(Z1)
Y = H @ W2 + b2
```

Z1은 은닉층에서 편향까지 더한 값, H는 ReLU를 적용한 값, Y는 최종 점수입니다. 은닉층에는 ReLU를 적용하고, 이 교안 예제의 출력층에는 별도 활성화 함수를 적용하지 않습니다.

#### 요구사항

1. 문제 2-1의 relu 함수를 그대로 작성하고, 위 입력과 파라미터를 만드세요.
2. Z1, H, Y를 제시된 순서대로 계산하여 출력하세요.
3. `H.shape`와 `Y.shape`를 출력하세요.
4. 고객별 H의 두 숫자를 사용해 최종 점수를 손으로 계산하세요.

#### 작성 도움말

W2의 첫 값은 2, 둘째 값은 −1입니다. 따라서 H의 한 행이 [h₁,h₂]이면 최종 점수는 `h₁×2 + h₂×(−1) + 1`입니다. h₁과 h₂는 각각 그 행의 첫째·둘째 숫자를 가리킵니다. 출력층에는 X나 Z1 대신 H를 넣으세요.

#### 해석 질문

**Q1.** Z1과 H는 어떻게 다른가요? 출력층의 입력으로 어느 것을 사용하나요?

**Q2.** 고객 A·B·C의 최종 점수는 각각 얼마인가요? 계산 과정을 적으세요.

**Q3.** H의 Shape (3,2)와 Y의 Shape (3,1)는 각각 무엇을 뜻하나요?

**Q4.** 고객 C의 최종 점수가 음수로 나와도 되는 이유는 무엇인가요?

#### 제출 결과

- Z1·H·Y 출력과 H·Y의 Shape
- 고객별 최종 계산 과정과 Q1~Q4 답변

In [10]:
# 문제 2-2. 은닉층의 결과를 출력층에 전달하기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.

def relu(z):
    return np.maximum(0, z)

X = np.array([
    [3,2],
    [1,4],
    [0,2]
])

W1 = np.array([
    [1,-2],
    [2,1]
])
b1 = np.array([-4,1])
W2 = np.array([
    [2],
    [-1]
])
b2 = np.array([1])

Z1 = X @ W1 + b1
H = relu(Z1)
Y = H @ W2 + b2
print("은닉층 편향 후 Z1:", Z1)
print("은닉층 출력 H:", H)
print("최종 출력 Y:", Y)
print("H shape:", H.shape)
print("Y shape:", Y.shape)

은닉층 편향 후 Z1: [[ 3 -3]
 [ 5  3]
 [ 0  3]]
은닉층 출력 H: [[3 0]
 [5 3]
 [0 3]]
최종 출력 Y: [[ 7]
 [ 8]
 [-2]]
H shape: (3, 2)
Y shape: (3, 1)


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. Z1과 H는 어떻게 다른가요? 출력층의 입력으로 어느 것을 사용하나요?**

- 답변: Z1은 은닉층에서 가중치를 곱해 더한 뒤 편향까지 더한 값이고, H는 Z1에 ReLU를 적용한 값이다
- 예를 들어 A의 Z1은 [3,-3], H는 [3,0]이다 출력층에는 ReLU 이후 값인 H를 사용한다

**Q2. 고객 A·B·C의 최종 점수는 각각 얼마인가요? 계산 과정을 적으세요.**

- 답변: A는 H=[3,0] 이므로 3*2+0*(-1)=6+0+1=7이다
- B는 H=[5,3] 이므로 5*2+3*(-1)=10-3+1=8이다
- C는 H=[0,3] 이므로 0*2+3*(-1)=0+(-3)+1=-2다
- 최종 점수는 고객 순서대로 7,8,-2

**Q3. H의 Shape (3,2)와 Y의 Shape (3,1)는 각각 무엇을 뜻하나요?**

- 답변: H의 (3,2)는 고객 3명의 은닉층 출력이 두 개씩 있다는 뜻이다
- Y의 (3,1)는 같은 고객 3명의 최종 출력이 한 개씩 있다는 뜻이다
- 행은 계속 고객을 나타내고 열은 그 단계에서 고객 당 얻는 숫자의 개수를 나타낸다

**Q4. 고객 C의 최종 점수가 음수로 나와도 되는 이유는 무엇인가요?**

- 답변: 이번 구조세어 은닉층에만 ReLU를 적용하고 출력층에는 적용하지 않기 떄문이다
- C의 H = [0,3]은 음수가 없지만 출력층에서 3에 -1을 곱하면 -3이 된다 편향 1을 더해도 -2가 남는다 -2는 계산 오류가 아니고 그대로 최종 점수다

### 문제 3-1. 선형 변환과 아핀 변환 다시 계산하기

#### 문제 설명

교안 연습문제 1입니다. 문제 1-1과 같은 방법을 사용하세요.

```python
x = np.array([2, 3])
W = np.array([[2, 0], [1, 3]])
bias = np.array([1, -2])
zero = np.array([0, 0])
```

W의 첫 열은 [2,1], 둘째 열은 [0,3]입니다. 이번에는 첫 열의 두 번째 가중치가 0이 아닌 1이라는 점을 확인하세요.

#### 요구사항

1. 위 배열을 만들고 `linear_output = x @ W`를 계산하여 출력하세요.
2. `affine_output = x @ W + bias`를 계산하여 출력하세요.
3. `zero @ W`와 `zero @ W + bias`를 출력하세요.
4. 각 출력값에 사용한 곱셈과 덧셈을 답변 작성란에 적으세요.

#### 작성 도움말

첫 출력은 `2×2 + 3×1`입니다. 둘째 출력은 둘째 열 [0,3]을 이용하세요. 선형 변환을 모두 구한 뒤 각 위치의 편향을 더합니다. 새로운 함수나 공식을 찾을 필요 없이 문제 1-1의 순서대로 계산하세요.

#### 해석 질문

**Q1.** 선형 변환의 첫 번째·두 번째 출력은 각각 어떻게 계산되나요?

**Q2.** 편향을 더한 아핀 변환 결과는 무엇인가요?

**Q3.** 0벡터 입력에서 선형 변환과 아핀 변환 결과는 각각 무엇인가요?

#### 제출 결과

- 선형·아핀 변환 및 0벡터의 출력
- 계산 과정과 Q1~Q3 답변

In [11]:
# 문제 3-1. 선형 변환과 아핀 변환 다시 계산하기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.

x = np.array([2, 3])
W = np.array([
    [2, 0], 
    [1, 3]
])
bias = np.array([1, -2])
zero = np.array([0, 0])

linear_output = x @ W
print("linear_output", linear_output)      # [7 9]

affine_output = x @ W + bias
print("affine_output", affine_output)      # [8 7]

print("zero @ W", zero @ W)                # [0 0]
print("zero @ W + bias", zero @ W + bias)  # [ 1 -2]

linear_output [7 9]
affine_output [8 7]
zero @ W [0 0]
zero @ W + bias [ 1 -2]


#### 답변 작성란

**수식 및 계산 과정**

- 작성:

**Q1. 선형 변환의 첫 번째·두 번째 출력은 각각 어떻게 계산되나요?**

- 답변: 첫 번째 출력은 x [2,3]과 W의 첫 열 [2,1]을 내적: 2 * 2 + 3 * 1 = 7
- 두 번째 출력은 x [2,3]과 W의 둘째 열 [0,3]을 내적: 2 * 0 + 3 * 3 = 9
- 선형 변환 결과는 [7, 9]
- 첫 열의 두 번째 가중치가 1이므로 첫 출력에 3 * 1 = 3이 더해진다

**Q2. 편향을 더한 아핀 변환 결과는 무엇인가요?**

- 답변: 선형 변환 결과 [7, 9]에 편향 [1, -2]를 위치별로 더한다
- 7 + 1 = 8, 9 + (-2) = 7
- 아핀 변환 결과는 [8, 7]

**Q3. 0벡터 입력에서 선형 변환과 아핀 변환 결과는 각각 무엇인가요?**

- 답변: 선형 변환 zero @ W는 0 * 2 + 0 * 1 = 0, 0 * 0 + 0 * 3 = 0이므로 [0, 0]
- 아핀 변환 zero @ W + bias는 [0, 0] + [1, -2] = [1, -2]
- 선형 변환은 0벡터를 항상 0벡터로 보낸다
- 아핀 변환은 편향 때문에 0벡터 입력에서도 편향 값 [1, -2]가 그대로 출력된다

### 문제 3-2. 같은 순서로 두 레이어 계산하기

#### 문제 설명

교안 연습문제 3입니다. 문제 2-2와 같은 순서를 적용합니다.

```python
X = np.array([[2, 1], [1, 3]])
W1 = np.array([[1, -1], [2, 1]])
b1 = np.array([-3, 0])
W2 = np.array([[2], [-1]])
b2 = np.array([1])
```

첫 행은 고객 A, 둘째 행은 고객 B입니다. 은닉층에서 ReLU를 적용하고, 출력층에는 별도 활성화 함수를 적용하지 않습니다.

#### 요구사항

1. 문제 2-1의 relu 함수를 작성하고 위 배열을 만드세요.
2. `Z1 = X @ W1 + b1`을 계산하여 출력하세요.
3. `H = relu(Z1)`을 계산하여 출력하세요.
4. `Y = H @ W2 + b2`와 `Y.shape`를 출력하세요.
5. 아래 질문 순서대로 두 고객의 계산 과정을 적으세요.

#### 작성 도움말

은닉층 첫 뉴런은 첫 열 [1,2]를 사용하고 편향 −3을 더합니다. 둘째 뉴런은 둘째 열 [−1,1]을 사용하고 편향 0을 더합니다. 그 다음 ReLU를 적용하세요. 출력층에서는 H의 첫 값×2 + 둘째 값×(−1) + 1을 계산합니다.

#### 해석 질문

**Q1.** 고객 A의 은닉층 Z1과 ReLU 결과 H는 각각 무엇인가요?

**Q2.** 고객 B의 은닉층 Z1과 ReLU 결과 H는 각각 무엇인가요?

**Q3.** 고객 A와 B의 최종 점수는 각각 얼마인가요? H의 값을 사용해 계산하세요.

**Q4.** 최종 출력의 Shape는 무엇이고, 각 숫자는 무엇을 뜻하나요?

#### 제출 결과

- Z1·H·Y 출력과 Y의 Shape
- 고객별 단계별 계산과 Q1~Q4 답변

In [12]:
# 문제 3-2. 같은 순서로 두 레이어 계산하기
# 위의 데이터와 작성 도움말을 보고 한 단계씩 코드를 작성하세요.
# 중간 결과를 출력하여 확인하세요.

def relu(z):
    return np.maximum(0, z)

X = np.array([[2, 1], [1, 3]])
W1 = np.array([[1, -1], [2, 1]])
b1 = np.array([-3, 0])
W2 = np.array([[2], [-1]])
b2 = np.array([1])

Z1 = X @ W1 + b1
print("Z1", Z1)         

H = relu(Z1)
print("H", H)            

Y = H @ W2 + b2
print("Y", Y)            
print("Y.shape", Y.shape)  

Z1 [[ 1 -1]
 [ 4  2]]
H [[1 0]
 [4 2]]
Y [[3]
 [7]]
Y.shape (2, 1)


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 고객 A의 은닉층 Z1과 ReLU 결과 H는 각각 무엇인가요?**

- 답변: 고객 A의 입력은 [2,1]
- 첫 뉴런: 2 * 1 + 1 * 2 = 4, 4 + (-3) = 1
- 둘째 뉴런: 2 * (-1) + 1 * 1 = -1, -1 + 0 = -1
- Z1 = [1, -1]
- ReLU 적용: 1은 그대로 1, -1은 음수이므로 0
- H = [1, 0]

**Q2. 고객 B의 은닉층 Z1과 ReLU 결과 H는 각각 무엇인가요?**

- 답변: 고객 B의 입력은 [1,3]
- 첫 뉴런: 1 * 1 + 3 * 2 = 7, 7 + (-3) = 4
- 둘째 뉴런: 1 * (-1) + 3 * 1 = 2, 2 + 0 = 2
- Z1 = [4, 2]
- ReLU 적용: 둘 다 양수라서 그대로
- H = [4, 2]

**Q3. 고객 A와 B의 최종 점수는 각각 얼마인가요? H의 값을 사용해 계산하세요.**

- 답변: 고객 A: H = [1, 0] → 1 * 2 + 0 * (-1) + 1 = 2 + 0 + 1 = 3
- 고객 B: H = [4, 2] → 4 * 2 + 2 * (-1) + 1 = 8 - 2 + 1 = 7
- 최종 점수는 A가 3, B가 7

**Q4. 최종 출력의 Shape는 무엇이고, 각 숫자는 무엇을 뜻하나요?**

- 답변: Y의 Shape는 (2,1)
- H (2,2) @ W2 (2,1)이므로 결과는 2행 1열
- 행은 고객(1행 A, 2행 B), 열은 출력 점수 1개를 뜻한다
- 3과 7은 두 레이어를 거쳐 계산한 고객별 최종 점수다
- 출력층에는 활성화 함수를 적용하지 않았으므로 확률이 아니라 계산된 점수 그대로다

---

## 실습 마무리

1. 선형 변환과 아핀 변환의 계산은 어떻게 다른가요?
2. 가중치와 편향은 각각 언제 사용하는 숫자인가요?
3. ReLU는 [−3,−1,0,2,5]를 어떻게 바꾸나요?
4. 두 레이어의 계산에서 출력층에 전달하는 값은 무엇인가요?
5. 최종 출력의 Shape (3,1)는 무엇을 뜻하나요?
6. 주어진 가중치와 편향으로 출력만 계산하면 학습이 이루어지나요?

### 실습 마무리 답변 작성란

**1. 선형 변환과 아핀 변환의 계산은 어떻게 다른가요?**

- 답변: 선형 변환은 x @ W 이다 아핀 변환은 그 결과에 편향을 더한 x @ W + b 이다
- 문제 1-1에서는 선형 결과 [6,6]에 편향 [1,-2]를 더해 아핀 결과 [7,4]를 얻었다 편향이 모두 0이면 계산 결과는 같다

**2. 가중치와 편향은 각각 언제 사용하는 숫자인가요?**

- 답변: 가중치는 각 입력에 곱하는 숫자고 편향은 곱셈 결과들을 더한 뒤 추가로 더하는 숫자다

**3. ReLU는 [−3,−1,0,2,5]를 어떻게 바꾸나요?**

- 답변: [0,0,0,2,5]이다 음수는 0으로 바꾸고 0과 양수는 그대로 둔다 배열에 숫자가 5개 있다는 사실은 바뀌지 않는다

**4. 두 레이어의 계산에서 출력층에 전달하는 값은 무엇인가요?**

- 답변: 은닉층의 가중합에 편향을 더한 Z1에 ReLU를 적용해 만든 H이다
- 순서는 Z1=X@W1+b1, H=relu(Z1), Y = H@W2 +b2

**5. 최종 출력의 Shape (3,1)는 무엇을 뜻하나요?**

- 답변: 3행 1열이라는 뜻이다 이 실습에서 각 행은 고객이므로 고객 3명에게 점수 한 개씩을 계산했다는 뜻이다

**6. 주어진 가중치와 편향으로 출력만 계산하면 학습이 이루어지나요?**

- 답변: 아니다 주어진 값으로 출력까지 계산하는 과정은 순전파다
- 이번 코드는 자동으로 고치지 않는다
- 출력 계산과 파라미터를 조정하는 학습을 구분해라